In [4]:
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
import wave
import os
import io
import pygame
import threading
import shutil
import time
import uuid
import subprocess
from PIL import Image, ImageTk

# --- 核心參數設定 ---
# 請務必確認 ffmpeg 執行檔路徑正確
FFMPEG_PATH = r"C:\Users\jt725\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.0.1-full_build\bin"
OUTPUT_BASE = r"C:\Users\jt725\test_ai_output"
os.environ['PATH'] = FFMPEG_PATH + os.pathsep + os.environ['PATH']

try:
    from pydub import AudioSegment as PyDubSegment
    PYDUB_AVAILABLE = True
except ImportError:
    PYDUB_AVAILABLE = False

# --- 全域變數管理 ---
G = {
    "root": None, 
    "frames": {}, 
    "status": None,
    "music_path": None,
    "is_playing": False,
    "total_length": 0,
    "temp_img": None,
    "start_time": 0
}

def set_status(msg):
    if G["status"]: G["status"].set(msg)

def show(name):
    G["frames"][name].tkraise()
    set_status(f"目前頁面：{name}")

# --- 1. AI 分音視窗類別 ---

class MusicFilterWindow:
    def __init__(self, parent):
        self.win = tk.Toplevel(parent)
        self.win.title("AI 音樂分音處理")
        self.win.geometry("600x580")
        
        # 初始化音訊設定
        if not pygame.mixer.get_init():
            pygame.mixer.init(frequency=44100, size=-16, channels=2, buffer=512)
        pygame.mixer.set_num_channels(10)
        
        self.input_wav = tk.StringVar(value=G["music_path"] if G["music_path"] else "")
        self.progress_val = tk.DoubleVar(value=0)
        self.is_processing = False
        self.current_pos = 0.0
        self.start_tick = 0
        self.is_playing = False
        self.total_duration = 0.0
        
        self.tracks = {
            "bass":   {"var": tk.BooleanVar(value=True), "file": "", "sound": None, "segment": None},
            "drums":  {"var": tk.BooleanVar(value=True), "file": "", "sound": None, "segment": None},
            "other":  {"var": tk.BooleanVar(value=True), "file": "", "sound": None, "segment": None},
            "vocals": {"var": tk.BooleanVar(value=True), "file": "", "sound": None, "segment": None}
        }
        self.setup_ui()

    def setup_ui(self):
        frame_file = ttk.LabelFrame(self.win, text=" 1. 選擇音樂 ", padding=10)
        frame_file.pack(fill="x", padx=10, pady=5)
        ttk.Entry(frame_file, textvariable=self.input_wav).pack(side="left", fill="x", expand=True)
        ttk.Button(frame_file, text="選取 WAV", command=self.select_file).pack(side="right", padx=5)

        frame_ai = ttk.LabelFrame(self.win, text=" 2. AI 樂器分離 (需安裝 Demucs) ", padding=10)
        frame_ai.pack(fill="x", padx=10, pady=5)
        self.device_var = tk.StringVar(value="cuda")
        device_box = ttk.Frame(frame_ai)
        device_box.pack()
        ttk.Radiobutton(device_box, text="GPU (CUDA)", variable=self.device_var, value="cuda").pack(side="left", padx=10)
        ttk.Radiobutton(device_box, text="CPU", variable=self.device_var, value="cpu").pack(side="left", padx=10)

        self.btn_start_ai = ttk.Button(frame_ai, text="啟動分離 (Demucs)", command=self.start_separation_thread)
        self.btn_start_ai.pack(pady=5)
        self.pb = ttk.Progressbar(frame_ai, variable=self.progress_val, maximum=100)
        self.pb.pack(fill="x", pady=5)
        self.lbl_status = ttk.Label(frame_ai, text="等待操作...")
        self.lbl_status.pack()

        frame_play = ttk.LabelFrame(self.win, text=" 3. 同步播放控制 ", padding=10)
        frame_play.pack(fill="x", padx=10, pady=5)
        self.lbl_time = ttk.Label(frame_play, text="00:00 / 00:00", font=("Consolas", 12))
        self.lbl_time.pack()
        self.play_pb = ttk.Progressbar(frame_play, orient="horizontal", length=400, mode="determinate")
        self.play_pb.pack(pady=5)
        
        btn_box = ttk.Frame(frame_play)
        btn_box.pack()
        ttk.Button(btn_box, text="<< 5s", command=lambda: self.seek(-5)).grid(row=0, column=0, padx=5)
        self.btn_play_sub = ttk.Button(btn_box, text="播放 / 暫停", command=self.toggle_play)
        self.btn_play_sub.grid(row=0, column=1, padx=5)
        ttk.Button(btn_box, text=">> 5s", command=lambda: self.seek(5)).grid(row=0, column=2, padx=5)

        frame_tracks = ttk.LabelFrame(self.win, text=" 4. 樂器音軌選擇 ", padding=10)
        frame_tracks.pack(fill="x", padx=10, pady=5)
        for name, data in self.tracks.items():
            ttk.Checkbutton(frame_tracks, text=name.capitalize(), variable=data["var"], 
                            command=self.on_track_change).pack(side="left", expand=True)

    def select_file(self):
        path = filedialog.askopenfilename(filetypes=[("WAV", "*.wav")])
        if path:
            self.input_wav.set(path)
            with wave.open(path, 'r') as f:
                self.total_duration = f.getnframes() / float(f.getframerate())
            self.stop_playback()

    def start_separation_thread(self):
        if not self.input_wav.get(): 
            messagebox.showwarning("提示", "請先選擇檔案")
            return
        if self.is_processing: return
        self.stop_playback()
        self.progress_val.set(0)
        threading.Thread(target=self.run_demucs, daemon=True).start()

    def run_demucs(self):
        self.is_processing = True
        self.btn_start_ai.config(state="disabled")
        self.lbl_status.config(text=f"AI 正在處理中 ({self.device_var.get().upper()})...")
        input_path = self.input_wav.get()
        temp_name = f"filter_temp_{uuid.uuid4().hex[:8]}.wav"
        try:
            shutil.copy(input_path, temp_name)
            job_output = os.path.join(OUTPUT_BASE, "filter_session")
            if os.path.exists(job_output): shutil.rmtree(job_output)
            os.makedirs(job_output, exist_ok=True)
            
            cmd = ['demucs', '-d', self.device_var.get(), '-n', 'htdemucs', temp_name, '-o', job_output]
            creation_flags = subprocess.CREATE_NO_WINDOW if os.name == 'nt' else 0
            proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, creationflags=creation_flags)
            
            while proc.poll() is None:
                proc.stdout.readline()
                curr = self.progress_val.get()
                if curr < 95: self.progress_val.set(curr + 0.5)
                self.win.update_idletasks()
                time.sleep(0.1)
                
            if proc.returncode == 0:
                self.progress_val.set(100)
                track_dir = os.path.join(job_output, "htdemucs", os.path.splitext(temp_name)[0])
                for name in self.tracks.keys():
                    p = os.path.join(track_dir, f"{name}.wav")
                    if os.path.exists(p): self.tracks[name]["file"] = p
                self._preload_audio()
                self.lbl_status.config(text="分離完成！可以開始播放。")
                messagebox.showinfo("成功", "樂器分離已完成")
            else:
                self.lbl_status.config(text="分離失敗，請檢查環境變數或改用 CPU。")
        except Exception as e:
            self.lbl_status.config(text=f"錯誤: {e}")
        finally:
            if os.path.exists(temp_name): os.remove(temp_name)
            self.is_processing = False
            self.btn_start_ai.config(state="normal")

    def _preload_audio(self):
        try:
            for name, data in self.tracks.items():
                if os.path.exists(data["file"]):
                    data["sound"] = pygame.mixer.Sound(data["file"])
                    if PYDUB_AVAILABLE:
                        data["segment"] = PyDubSegment.from_wav(data["file"])
        except Exception as e:
            print(f"預載入失敗: {e}")

    def toggle_play(self):
        if self.is_playing: self.pause_playback()
        else: self.start_playback()

    def start_playback(self):
        pygame.mixer.stop()
        time.sleep(0.05)
        start_ms = int(self.current_pos * 1000)
        success = False
        for i, (name, data) in enumerate(self.tracks.items()):
            if not data["var"].get(): continue
            chan = pygame.mixer.Channel(i)
            try:
                if PYDUB_AVAILABLE and data["segment"] and start_ms > 0:
                    part = data["segment"][start_ms:]
                    fp = io.BytesIO()
                    part.export(fp, format="wav")
                    fp.seek(0)
                    chan.play(pygame.mixer.Sound(fp))
                    success = True
                elif data["sound"]:
                    chan.play(data["sound"])
                    success = True
            except: pass
        if success:
            self.is_playing = True
            self.start_tick = time.time()
            self.run_sync_loop()

    def pause_playback(self):
        if self.is_playing:
            self.current_pos += (time.time() - self.start_tick)
            pygame.mixer.stop()
            self.is_playing = False

    def stop_playback(self):
        pygame.mixer.stop()
        self.is_playing = False
        self.current_pos = 0
        self.update_ui_display(0)

    def on_track_change(self):
        if self.is_playing:
            self.current_pos += (time.time() - self.start_tick)
            self.start_playback()

    def seek(self, seconds):
        if self.is_playing: self.current_pos += (time.time() - self.start_tick)
        self.current_pos = max(0, min(self.current_pos + seconds, self.total_duration))
        if self.is_playing: self.start_playback()
        else: self.update_ui_display(self.current_pos)

    def run_sync_loop(self):
        if self.is_playing:
            now_pos = self.current_pos + (time.time() - self.start_tick)
            if now_pos >= self.total_duration: self.stop_playback()
            else:
                self.update_ui_display(now_pos)
                self.win.after(50, self.run_sync_loop)

    def update_ui_display(self, pos):
        cur_m, cur_s = divmod(int(pos), 60)
        tot_m, tot_s = divmod(int(self.total_duration), 60)
        self.lbl_time.config(text=f"{cur_m:02d}:{cur_s:02d} / {tot_m:02d}:{tot_s:02d}")
        if self.total_duration > 0: self.play_pb['value'] = (pos / self.total_duration) * 100

# --- 2. 核心邏輯 (隱寫術) ---

def hide_image_logic(wav_path, img_path, save_path):
    try:
        with Image.open(img_path) as img:
            img_byte_arr = io.BytesIO()
            img.save(img_byte_arr, format='PNG')
            img_data = img_byte_arr.getvalue()
        data_to_hide = len(img_data).to_bytes(4, byteorder='big') + img_data + b"###"
        binary_msg = ''.join([format(b, '08b') for b in data_to_hide])
        with wave.open(wav_path, 'rb') as audio:
            params = audio.getparams()
            frame_bytes = bytearray(list(audio.readframes(audio.getnframes())))
        if len(binary_msg) > len(frame_bytes): return False, "空間不足，音樂太短或圖片太大！"
        for i in range(len(binary_msg)):
            frame_bytes[i] = (frame_bytes[i] & 254) | int(binary_msg[i])
        with wave.open(save_path, 'wb') as f:
            f.setparams(params)
            f.writeframes(bytes(frame_bytes))
        return True, save_path
    except Exception as e: return False, str(e)

def extract_image_logic(wav_path):
    try:
        with wave.open(wav_path, 'rb') as audio:
            frame_bytes = bytearray(list(audio.readframes(audio.getnframes())))
        def get_byte_at(start):
            bits = "".join([str(frame_bytes[j] & 1) for j in range(start, start + 8)])
            return int(bits, 2)
        len_bytes = bytearray([get_byte_at(i) for i in range(0, 32, 8)])
        img_len = int.from_bytes(len_bytes, byteorder='big')
        img_data = bytearray([get_byte_at(i) for i in range(32, 32 + img_len * 8, 8)])
        return Image.open(io.BytesIO(img_data))
    except: return None

# --- 3. UI 頁面與主控制 ---

def main_toggle_play(pb, lbl):
    if not G["music_path"]: return
    if not pygame.mixer.music.get_busy() and not G["is_playing"]:
        pygame.mixer.music.load(G["music_path"])
        pygame.mixer.music.play(start=G["start_time"])
        G["is_playing"] = True
        audio = wave.open(G["music_path"])
        G["total_length"] = audio.getnframes() / audio.getframerate()
        audio.close()
        main_update_progress(pb, lbl)
        set_status("播放中...")
    elif G["is_playing"]:
        G["start_time"] += (pygame.mixer.music.get_pos() / 1000.0)
        pygame.mixer.music.stop()
        G["is_playing"] = False
        set_status("已暫停")

def main_update_progress(pb, lbl):
    if G["is_playing"]:
        curr = G["start_time"] + (pygame.mixer.music.get_pos() / 1000.0)
        if curr <= G["total_length"]:
            pb['value'] = (curr / G["total_length"]) * 100
            mins, secs = divmod(int(curr), 60)
            lbl.config(text=f"▶︎ • {mins:02d}:{secs:02d}")
            G["root"].after(500, lambda: main_update_progress(pb, lbl))
        else:
            G["is_playing"] = False
            G["start_time"] = 0
            set_status("播放結束")

def main_seek(sec):
    if not G["music_path"]: return
    cur = G["start_time"] + (pygame.mixer.music.get_pos() / 1000.0)
    G["start_time"] = max(0, min(cur + sec, G["total_length"]))
    pygame.mixer.music.play(start=G["start_time"])

def frame_operation(parent):
    f = ttk.Frame(parent)
    ttk.Label(f, text="操作與檢視", font=("Microsoft JhengHei", 16, "bold")).pack(pady=10)
    lbl_file = ttk.Label(f, text="尚未選擇音樂")
    lbl_file.pack()
    lbl_time = ttk.Label(f, text="▶︎ • 00:00", font=("Consolas", 14))
    lbl_time.pack(pady=5)
    pb = ttk.Progressbar(f, orient="horizontal", length=300, mode="determinate")
    pb.pack()

    btn_box = ttk.Frame(f)
    btn_box.pack(pady=10)

    def select_wav():
        p = filedialog.askopenfilename(filetypes=[("WAV", "*.wav")])
        if p:
            G["music_path"] = p
            lbl_file.config(text=os.path.basename(p))

    ttk.Button(btn_box, text="選擇 WAV", command=select_wav).grid(row=0, column=0, padx=5)
    ttk.Button(btn_box, text="<<", width=4, command=lambda: main_seek(-5)).grid(row=0, column=1, padx=2)
    ttk.Button(btn_box, text="▶︎", width=4, command=lambda: main_toggle_play(pb, lbl_time)).grid(row=0, column=2, padx=2)
    ttk.Button(btn_box, text=">>", width=4, command=lambda: main_seek(5)).grid(row=0, column=3, padx=2)
    
    # [分音] 按鈕與原本按鈕並列
    ttk.Button(btn_box, text="分音", command=lambda: MusicFilterWindow(G["root"])).grid(row=0, column=4, padx=5)
    
    def show_score():
        img = extract_image_logic(G["music_path"])
        if img:
            win = tk.Toplevel(G["root"])
            win.title("樂譜檢視")
            tk_img = ImageTk.PhotoImage(img)
            G["temp_img"] = tk_img
            tk.Label(win, image=tk_img).pack()
        else: messagebox.showerror("錯誤", "找不到譜")
        
    ttk.Button(f, text="打開隱藏譜", command=show_score).pack(pady=5)
    return f

# --- 其餘頁面與主程式 ---

def frame_intro(parent):
    f = ttk.Frame(parent)
    ttk.Label(f, text="音樂譜隱藏系統", font=("Microsoft JhengHei", 24, "bold")).pack(pady=30)
    ttk.Button(f, text="進入系統", command=lambda: show("操作")).pack()
    return f

def frame_hide(parent):
    f = ttk.Frame(parent)
    ttk.Label(f, text="隱藏樂譜功能", font=("Microsoft JhengHei", 16, "bold")).pack(pady=20)
    m_p, i_p = tk.StringVar(value="..."), tk.StringVar(value="...")
    ttk.Label(f, textvariable=m_p).pack()
    ttk.Button(f, text="選擇音樂", command=lambda: m_p.set(filedialog.askopenfilename() or "...")).pack()
    ttk.Label(f, textvariable=i_p).pack()
    ttk.Button(f, text="選擇圖片", command=lambda: i_p.set(filedialog.askopenfilename() or "...")).pack()
    def do_hide():
        save = filedialog.asksaveasfilename(defaultextension=".wav")
        if save:
            suc, res = hide_image_logic(m_p.get(), i_p.get(), save)
            if suc: 
                G["music_path"] = res
                show("操作")
    ttk.Button(f, text="放入音樂", command=do_hide).pack(pady=20)
    return f

def build_menu():
    mbar = tk.Menu(G["root"])
    m = tk.Menu(mbar, tearoff=0)
    m.add_command(label="操作頁", command=lambda: show("操作"))
    m.add_command(label="隱藏樂譜", command=lambda: show("隱藏"))
    m.add_command(label="分音視窗", command=lambda: MusicFilterWindow(G["root"]))
    m.add_separator()
    m.add_command(label="離開", command=G["root"].quit)
    mbar.add_cascade(label="功能", menu=m)
    G["root"].config(menu=mbar)

def main():
    pygame.mixer.init()
    root = tk.Tk()
    root.title("音樂譜隱藏與分音系統")
    root.geometry("600x400")
    G["root"] = root
    build_menu()
    container = ttk.Frame(root)
    container.pack(fill="both", expand=True)
    G["frames"]["說明"] = frame_intro(container)
    G["frames"]["操作"] = frame_operation(container)
    G["frames"]["隱藏"] = frame_hide(container)
    for fr in G["frames"].values(): fr.place(x=0, y=0, relwidth=1, relheight=1)
    G["status"] = tk.StringVar(value="就緒")
    ttk.Label(root, textvariable=G["status"], anchor="w").pack(fill="x", side="bottom")
    show("說明")
    root.mainloop()

if __name__ == "__main__":
    main()

pydub 載入成功
